In [35]:
# Imports 
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from scipy.signal import resample
import sqlite3

In [36]:
# Load data from MediaPipe Pose stored in Parquet file

df = pl.read_parquet('/Users/pierre/Documents/NF_Bootcamp/Capstone/GAITy-Capstone-Modeling/data/filled_gait_data_encoded.parquet')


In [ ]:
"""mg_meta = pl.read_csv(
    "/Users/pierre/Documents/NF_Bootcamp/Capstone/GAITy-Capstone-Modeling/data/merged_summary_enriched_full_final_2_b.csv",
    separator=";",
    infer_schema_length=10_000,
    truncate_ragged_lines=True
)"""

In [ ]:
"""mg_meta_sel = mg_meta.select([
    "patient_name",
    "video_speed",
])"""

In [ ]:
"""df = (
    df.join(
        mg_meta_sel,
        on="patient_name",
        how="left"
    )
    .with_columns(
        pl.col("video_speed").fill_null("N")
    )
)"""


In [37]:
df.tail(1000).to_pandas()

,id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,...,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,file_path,video_id,dataset_encoded
0,21897404,cllbjuuah00133o6l8feqshab,228,N,left side,3800.0,29,0.403027,0.895700,0.075293,...,abnormal,normal,Case Study 11 - 6 Weeks Later,60.0,1920,1080,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,cllbjuuah00133o6l8feqshab_right side_nan_Abnor...,1
1,21897405,cllbjuuah00133o6l8feqshab,228,N,left side,3800.0,30,0.597519,0.827139,0.249480,...,abnormal,normal,Case Study 11 - 6 Weeks Later,60.0,1920,1080,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,cllbjuuah00133o6l8feqshab_right side_nan_Abnor...,1
2,21897406,cllbjuuah00133o6l8feqshab,228,N,left side,3800.0,31,0.332590,0.892155,0.003448,...,abnormal,normal,Case Study 11 - 6 Weeks Later,60.0,1920,1080,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,cllbjuuah00133o6l8feqshab_right side_nan_Abnor...,1
3,21897407,cllbjuuah00133o6l8feqshab,228,N,left side,3800.0,32,0.544770,0.875209,0.203711,...,abnormal,normal,Case Study 11 - 6 Weeks Later,60.0,1920,1080,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,cllbjuuah00133o6l8feqshab_right side_nan_Abnor...,1
4,21897410,cllbjuuah00133o6l8feqshab,229,N,left side,3816.0,2,0.426575,0.002124,-0.137103,...,abnormal,normal,Case Study 11 - 6 Weeks Later,60.0,1920,1080,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,cllbjuuah00133o6l8feqshab_right side_nan_Abnor...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,21899748,cllbjuuah00133o6l8feqshab,299,N,left side,4983.0,30,0.521933,0.775086,0.327200,...,abnormal,normal,Case Study 11 - 6 Weeks Later,60.0,1920,1080,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,cllbjuuah00133o6l8feqshab_right side_nan_Abnor...,1
996,21899749,cllbjuuah00133o6l8feqshab,299,N,left side,4983.0,31,0.379858,0.914215,-0.032618,...,abnormal,normal,Case Study 11 - 6 Weeks Later,60.0,1920,1080,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,cllbjuuah00133o6l8feqshab_right side_nan_Abnor...,1
997,21899750,cllbjuuah00133o6l8feqshab,299,N,left side,4983.0,32,0.472536,0.857770,0.261948,...,abnormal,normal,Case Study 11 - 6 Weeks Later,60.0,1920,1080,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,cllbjuuah00133o6l8feqshab_right side_nan_Abnor...,1
998,21899753,cllbjuuah00133o6l8feqshab,300,N,left side,5000.0,2,0.406752,-0.025397,-0.156729,...,abnormal,normal,Case Study 11 - 6 Weeks Later,60.0,1920,1080,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,cllbjuuah00133o6l8feqshab_right side_nan_Abnor...,1


In [38]:
df.group_by("movement_type").count()


/var/folders/0b/_k7dtrz57nd823f3sv2whm600000gn/T/ipykernel_17404/1418624679.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  df.group_by("movement_type").count()


movement_type,count
str,u32
null,260750
"""until second 6, then slowow""",8960
"""Regular Movement""",4538198
"""SLOWMOTION""",866642
"""Fast Movement""",3069332
"""N""",507822


In [39]:
#Fill null values with "N"
df = df.with_columns(
    pl.col("movement_type").fill_null("N")
)

In [40]:
df.group_by("movement_type").count()

/var/folders/0b/_k7dtrz57nd823f3sv2whm600000gn/T/ipykernel_17404/1518174712.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  df.group_by("movement_type").count()


movement_type,count
str,u32
"""N""",768572
"""Fast Movement""",3069332
"""until second 6, then slowow""",8960
"""SLOWMOTION""",866642
"""Regular Movement""",4538198


In [41]:
# Remove Slowmotion videos and keep any null values
df = df.filter(
    (pl.col("movement_type").is_null()) | 
    (~pl.col("movement_type").str.contains("SLOWMOTION"))
)


In [33]:
df.group_by("movement_type").count()

/var/folders/0b/_k7dtrz57nd823f3sv2whm600000gn/T/ipykernel_17404/1518174712.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  df.group_by("movement_type").count()


movement_type,count
str,u32
"""Fast Movement""",3069332
"""Regular Movement""",4538198
"""N""",507822
"""until second 6, then slowow""",8960


In [5]:
df.group_by("dataset").count()


/var/folders/0b/_k7dtrz57nd823f3sv2whm600000gn/T/ipykernel_17404/4100980092.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  df.group_by("dataset").count()


dataset,count
str,u32
"""normal""",7607530
"""abnormal""",1644174


In [7]:
df.group_by("dataset").agg(
    pl.col("video_id").n_unique().alias("n_videos")
)


dataset,n_videos
str,u32
"""abnormal""",212
"""normal""",3067


In [12]:
df.group_by("dataset").agg(
    pl.struct(["video_id", "frame"])
      .n_unique()
      .alias("n_frames")
)



dataset,n_frames
str,u32
"""normal""",543395
"""abnormal""",117441


In [43]:
df.null_count()

id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,visibility,x_px,y_px,dataset,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,file_path,video_id,dataset_encoded
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7968940,0,0,0


In [44]:
df.group_by("dataset").agg(
    pl.len().alias("n_entries"),
    pl.struct(["video_id", "frame"]).n_unique().alias("n_frames"),
    pl.col("video_id").n_unique().alias("n_videos"),
)

dataset,n_entries,n_frames,n_videos
str,u32,u32,u32
"""normal""",7607530,543395,3067
"""abnormal""",777532,55538,112


In [ ]:
print(df.shape) 
print(df.columns)
print(df.dtypes)

In [ ]:
# Sort by video_id, then id
df_s = df.sort(["video_id", "id"], 
               descending=[False, False])
#Add new Index
df_s = df_s.with_columns(pl.arange(0,df_s.height).alias("new_index"))
# Move new_Index to the first column
df_s = df_s.select(["new_index"] + [c for c in df_s.columns if c != "new_index"])

In [ ]:
df_s.head(1000).to_pandas()

In [ ]:
import polars as pl
import numpy as np

def missing_frames_summary_polars(df: pl.DataFrame, video_col="video_id", frame_col="frame"):
    """
    Compute missing frames statistics per video with mean, std, max, and % missing.
    Returns per-video stats sorted by number of missing frames (descending).
    """

    # Step 1: Count unique frames per video
    unique_frame_counts = (
        df.group_by(video_col)
          .agg(pl.col(frame_col).n_unique().alias("num_unique_frames"))
    )

    # Step 2: Compute min/max frame per video
    frame_range = (
        df.group_by(video_col)
          .agg([
              pl.col(frame_col).min().alias("min_frame"),
              pl.col(frame_col).max().alias("max_frame")
          ])
    )

    # Step 3: Join counts with ranges
    frame_stats = frame_range.join(unique_frame_counts, on=video_col)

    # Step 4: Compute missing frames
    frame_stats = frame_stats.with_columns([
        (pl.col("max_frame") - pl.col("min_frame") + 1 - pl.col("num_unique_frames"))
        .alias("num_missing_frames")
    ])

    # Step 5: Compute missing frames as percentage
    frame_stats = frame_stats.with_columns([
        ((pl.col("num_missing_frames") / (pl.col("max_frame") - pl.col("min_frame") + 1)) * 100)
        .alias("pct_missing")
    ])

    # Step 6: Sort by number of missing frames descending
    frame_stats = frame_stats.sort("num_missing_frames",descending=True)

    # Step 7: Convert to NumPy arrays for summary statistics
    num_missing_array = frame_stats["num_missing_frames"].to_numpy()
    pct_missing_array = frame_stats["pct_missing"].to_numpy()

    # Step 8: Compute summary
    summary = {
        "num_missing_frames_mean": np.mean(num_missing_array),
        "num_missing_frames_std": np.std(num_missing_array),
        "num_missing_frames_max": np.max(num_missing_array),
        "pct_missing_mean": np.mean(pct_missing_array),
    }

    return summary, frame_stats



In [ ]:
summary, frame_stats = missing_frames_summary_polars(df_s, video_col="video_id", frame_col="frame")

print("=== Missing Frames Summary ===")
for k, v in summary.items():
    print(f"{k}: {v:.2f}")

# The per-video stats are now sorted by number of missing frames descending
print(frame_stats)


=== Missing Frames Summary ===
num_missing_frames_mean: 0.40
num_missing_frames_std: 2.00
num_missing_frames_max: 57.00
pct_missing_mean: 0.19